# Optimization Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Define a test function

The Rosenbrock function is a classic optimization benchmark. Its minimum is at (1, 1) inside a narrow curved valley that is easy to find but hard to follow.

In [ ]:
```

f(x, y) = (1 - x)^2 + 100 * (y - x^2)^2

In [ ]:
```

In [ ]:
```python

def rosenbrock(params):

    x, y = params

    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def rosenbrock_gradient(params):

    x, y = params

    df_dx = -2 * (1 - x) + 200 * (y - x ** 2) * (-2 * x)

    df_dy = 200 * (y - x ** 2)

    return [df_dx, df_dy]

In [ ]:
```

### Step 2: Vanilla gradient descent

In [ ]:
```python

class GradientDescent:

    def __init__(self, lr=0.001):

        self.lr = lr

    def step(self, params, grads):

        return [p - self.lr * g for p, g in zip(params, grads)]

In [ ]:
```

### Step 3: SGD with momentum

In [ ]:
```python

class SGDMomentum:

    def __init__(self, lr=0.001, momentum=0.9):

        self.lr = lr

        self.momentum = momentum

        self.velocity = None

    def step(self, params, grads):

        if self.velocity is None:

            self.velocity = [0.0] * len(params)

        self.velocity = [

            self.momentum * v + g

            for v, g in zip(self.velocity, grads)

        ]

        return [p - self.lr * v for p, v in zip(params, self.velocity)]

In [ ]:
```

### Step 4: Adam

In [ ]:
```python

class Adam:

    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):

        self.lr = lr

        self.beta1 = beta1

        self.beta2 = beta2

        self.epsilon = epsilon

        self.m = None

        self.v = None

        self.t = 0

    def step(self, params, grads):

        if self.m is None:

            self.m = [0.0] * len(params)

            self.v = [0.0] * len(params)

        self.t += 1

        self.m = [

            self.beta1 * m + (1 - self.beta1) * g

            for m, g in zip(self.m, grads)

        ]

        self.v = [

            self.beta2 * v + (1 - self.beta2) * g ** 2

            for v, g in zip(self.v, grads)

        ]

        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]

        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]

        return [

            p - self.lr * mh / (vh ** 0.5 + self.epsilon)

            for p, mh, vh in zip(params, m_hat, v_hat)

        ]

In [ ]:
```

### Step 5: Run and compare

In [ ]:
```python

def optimize(optimizer, func, grad_func, start, steps=5000):

    params = list(start)

    history = [params[:]]

    for _ in range(steps):

        grads = grad_func(params)

        params = optimizer.step(params, grads)

        history.append(params[:])

    return history

start = [-1.0, 1.0]

gd_history = optimize(GradientDescent(lr=0.0005), rosenbrock, rosenbrock_gradient, start)

sgd_history = optimize(SGDMomentum(lr=0.0001, momentum=0.9), rosenbrock, rosenbrock_gradient, start)

adam_history = optimize(Adam(lr=0.01), rosenbrock, rosenbrock_gradient, start)

for name, history in [("GD", gd_history), ("SGD+M", sgd_history), ("Adam", adam_history)]:

    final = history[-1]

    loss = rosenbrock(final)

    print(f"{name:6s} -> x={final[0]:.6f}, y={final[1]:.6f}, loss={loss:.8f}")

In [ ]:
```

Expected output: Adam converges fastest. SGD with momentum follows a smoother path. Vanilla GD makes slow progress along the narrow valley.

## Exercises

In [ ]:
1. **Learning rate sweep.** Run vanilla gradient descent on the Rosenbrock function with learning rates [0.0001, 0.0005, 0.001, 0.005, 0.01]. Plot or print the final loss after 5000 steps for each. Find the largest learning rate that still converges.

2. **Momentum comparison.** Run SGD with momentum values [0.0, 0.5, 0.9, 0.99] on the Rosenbrock function. Track the loss at every step. Which momentum value converges fastest? Which overshoots?

3. **Saddle point escape.** Define the function `f(x, y) = x^2 - y^2` (a saddle point at the origin). Start at (0.01, 0.01). Compare how vanilla GD, SGD with momentum, and Adam behave. Which escapes the saddle point?

4. **Implement learning rate decay.** Add an exponential decay schedule to the GradientDescent class: `lr = lr_0 * 0.999^step`. Compare convergence with and without decay on the Rosenbrock function.